CASO PRATICO: ARCHITETTURA E SVILUPPO DI UN CHATBOT SEMPLICE

Un chatbot semplice è un caso pratico che mette insieme quasi tutto quello visto finora: NLP, tokenizzazione, classificazione, Transformer e generazione testo. 
Chatbot non identifica una particolare rete neurale, è un'applicazione compostas di più pezzi:
UTENTE -> "frase (prompt) -> preparazione del testo -> modello NLP/LLM -> interpretazione richiesta -> logica applicativa -> generazione/selezione della risposta -> chatbot -> "risposta"

Esistono 3 tipi di chatbot:
1) Chatbot a regole: non utilizza l'ai è qualosa del tipo: if utente scrive "ciao" rispondi "ciao come stai?" input - regole - output. Va bene per pochissime richiesta prevedibili, ma appena l'utente cambia frase le regole cominciano a diventare difficili da gestire.
2) Chatbot basato sulla classificazione di intenti: supponiamo di avere questi intenti: saluto, orario, stato_ordine, reclamo, richiesta_offerta; l'utente scrive: "quando arriverà il mio ordine n 1287?", il chatbot fa: classificazione (stato_ordine) e NER (numero= 1287), l'applicazione interroga il database e risponde.La classificazione potrebbe essere fata con: TF-IDF + Logistic Regression, TF-IDF + SVM, BERT fine-tuned, Transformer; mentre l'estrazione dell'informazione con NER, regex, spaCy, BERT fine-tuned per NER
3) Chatbot generativo: simile a ChatGPT, potrebbe essere: messaggio utente (prompt) - tokenizer - token IDs - modello GPT - generate() - token IDs risposta - decode - Testo di risostsa. C'è però un altro componente fondamentale, la cronologia della conversazione, qui entrano in gioco le tecniche per il mantenimento del contesto.

Come facccimo a cosa vuole davvero l'utente?

Intenti e Modelli Generativi
Definire lo scopo dell'utente
Il primo combito di un chatbot moderno non è parlaare, ma ascltare, non si limita a rispondere, ma cerca inanzitutto di capire cosa l'utente desidera fare attraverso l'intent Recognition.
Se l'utente scrive "voglio prenotare un tavolo" il chatbot devi prima individuare lo scopo "intent recognition"
classificatori di intenti + modelli casuali come GPT-2 = risposte naturali

Ma come si scompone questa capacità di comprensione?

Anatomia della  Comprensione
Dall'input alla funzione di risposta
- Intent Classification (classificazione degli intenti): mappare l'input dell'utente in categorie predefinite (es. "saluto", "info_prodotto", "reset")
- Slot Filling: estrazione di entità specifiche necessarie per completare l'azione richiesta dall'utente. Se l'utente vuole prenotare, dove e quando sono i bagagli che deve portare con se e noi dobbiamo estrarli.
- LLM Integration: utilizzo dei modelli di linguaggio per gestire la 'coda lunga' della domande non previste dagli intenti fissi. L'integrazione con LLM è utile quando l'utente fa una domanda fuori 'standard'
- Fallback Strategy: gestione dei casi in cui il modello non è sicuro dell'intenzione dell'utente. Un modo per dire 'scusa non ho capito puoi ripetere?' Come avere una rete di sicurezza per non far cadere la conversazione.

Quindi serve un modello che classifica o uno che genera?

La risposta è: entrambi

Mappare il Dialogo
I modelli basati su intenti (classificatori) sono deterministici e sicuri, stabili, veloci e prevedibli, ideali per funzioni critiche come i pagamenti;
I modelli generativi sono creativi ma meno prevedibili, flessibili e naturali.
Per evitare che il nostro chatbot inventi troppo, dobbiamo collegarlo ad una Knowledge Base, una sorgenti di verità (attinge informazioni per evitare di inventare informazioni durante la generazione)
Infine, ogni predizione di intento è associata a una probabilità che determina se attivare una risposta automatica o chiedere chiarimenti (confidence score). Se la probabilità del classificatore è bassa, non tiriamo ad indovinare, ma chiediamo conferma all'utente o passiamo la palla al modello generativo.

Matemtica dell'Intento
Softmax over Intents
La classificazione dell'intento i dato l'input u viene calcolata normalizzando i punteggi di uscita del layer finale della rete, tramite la funzione softmax.
Questo permette di impostare una soglia minima (threshold) per la validazione della risposta.
Questa soglia è il cuore che decide se il  bot debba rispondere con una template fisso o tentare una generazione più complessa.

Come facciamo a ricordare l'intera storia?

Entriamo nel campo del mantenimento del contesto.
Un chatbot efficace deve ricordare ciò che è stato detto nei turni precedenti.
Dobbiamo implementare un buffer di memoria (memoria a breve termine) che scorre insieme alla conversazione per nutrire il modello di linguaggio.

Quali tecniche possiamo usare per gestire questa memoria?

Gestione dello Stato
Tecniche di presistenza del dialogo
- Conversational Buffer: una coda che mantiene gli ultimi N scambi tra utente e bot
- Context Window: il limite fisico di token che il Transformer può processare in una singola chiamata. cioè i transformer hanno un numero massimo di parole che possono leggere in un colpo solo. Non possiamo dare al bot un intero archivio di 1 anno di chat
- History Truncation: rimozione dei messaggi più vecchi per far spazio a quelli nuovi senza rompere la coerenza.
- State Tracking: variabile che tiene traccia di informazioni chiave (es. nome utente) durante tutta la sessione. Li salviamo in un assetes separato che non vengano perso nel taglio delle cronologie (history truncation)

Esistono modi più intelligenti per non dimenticare le cose importanti?
Si, possiamo usare la Context Summarization, se la storia è troppo lunga, invece di tagliare i vecchi messaggi, chiediamo ad un altro modelli di riassumerli prima di passarli al bot.
E' fondamentale distinguere nel buffer, cosa ha detto l''user' e cosa ha risposto l'assistant' tramite tag specifici (role tagging). Se scrivo solo il testo il bot potrebbe confondersi e pensare di aver detto lui quello che ha detto l'utente
Usiamo la sliding windows, una finestra mobile che scorre sulla conversazione mantenendo l'attenzione sul presente senza perdere il filo conduttore immediato.

Questa storia come viene rappresentata numericamente per la rete?
Rappresentazine del Contesto, vettore di storia
La storia è una sequenza che contiene gli ultimi k scambi tra utente e bot. Questa strututra permette al meccanismo di attenzione, di pesare le parole passata e dare loro la giusta importanza nel generare la risposta corrente.

Ciclo di Vita del Bot
Struttura dello script Python (app.py):
- Inizialization: caricamento del modello e del tokenizer (es. GPT-2 via Hugging Face)
- Main loop: un ciclo 'while True' che cattura l'input dell'utente finchè non riceve un comando di uscita
- Inference Step: tokenizzazione, generazione del testo e decodifica dell'output in stringa leggibile
- Post-processing: pulizia del testo generato per rimuovere ripetizioni o tag tecnici non necessari. BERT e GPT-2 a volte generano spazzi strani o dettagli tecnici, dobbiamo pulire la risposta prima di mostrarla all'utente per farla sembrare naturale.

Ci sono dei dettagli implementativi che fanno la differenza tra un bot mediocre ed uno professionale.
Dettagli Implementativi, dobbiamo pensare all'esperienza utente.
- Comandi speciali: gestione di parole chiave come 'exit' o 'quit' per chiudere la sessione in modo elegante
- User Experience: aggiunta di indicatori visivi (es. 'Bot: ...') per rendere la chat leggibile nel terminale. Indichiamo chiaramente chi sta parlando.
- Error Handling: prevenire crash se l'utente invia stringhe vuote o caratteri non supportati. magari rispondere con una risosta 'simpatica'

Ma quanto tempo ci mette il bot a rispondere?

Efficienza Temporale
La latenza è il nemico numero 1, il tempo toale è la somma del tempo di preprocessing, del tempo di inferenza e del post processing, monitorare questo valore ci permette di capire se usare un modello più piccolo

In [ ]:
"""
================================================================================
ESEMPIO PRATICO: ARCHITETTURA DI UN CHATBOT CON MEMORIA (BACKEND PYTORCH)
================================================================================
In questo script implementiamo un chatbot utilizzando la libreria 'transformers'
di Hugging Face, che è lo standard industriale nel 2026.

Il codice dimostra tre concetti chiave:
1. TOKENIZZAZIONE: Trasformazione del testo in numeri comprensibili ai neuroni.
2. CONTEXT BUFFER: Come mantenere la "memoria" degli ultimi scambi.
3. INFERENZA CAUSALE: La generazione di nuove parole basata sul contesto passato.
"""

import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- CONFIGURAZIONE ---
# Specifichiamo che il backend preferito è PyTorch. 
# Molte librerie moderne come Keras 3 leggono questa variabile d'ambiente.
os.environ["KERAS_BACKEND"] = "torch"

def initialize_chatbot():
    """
    Inizializza i componenti core dell'IA.
    
    1. Tokenizer: Colui che traduce le parole in ID numerici.
    2. Model: Il cervello (GPT-2) che processa i numeri e genera altri numeri.
    """
    print("\n[STEP 1]: Risveglio dei neuroni (Caricamento GPT-2 via Hugging Face)...")
    
    # Usiamo 'gpt2', un modello bilanciato per dimostrazioni didattiche.
    model_name = "gpt2"
    
    # Carichiamo il 'traduttore' (Tokenizer)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    # Carichiamo il 'cervello' (Model) predisposto per la generazione di testo (CausalLM)
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # GPT-2 non ha un pad_token di default. Lo impostiamo uguale all'eos_token (End Of Sentence)
    # per evitare errori durante l'elaborazione di sequenze di lunghezza diversa.
    tokenizer.pad_token = tokenizer.eos_token
    
    return model, tokenizer

def chat_loop():
    """
    Gestisce il ciclo interattivo, la memoria a breve termine e la generazione.
    """
    # Fase di setup iniziale
    model, tokenizer = initialize_chatbot()
    
    # BUFFER DI MEMORIA: Una lista che funge da 'coda' per i ricordi.
    history_buffer = [] 
    max_history = 3 # Numero massimo di messaggi da 'ricordare' (Sliding Window)

    print("\n" + "="*50)
    print(" CHATBOT ATTIVO (Versione Windows/Torch) ".center(50, "="))
    print("="*50)

    while True:
        # CATTURA INPUT: Leggiamo cosa scrive l'utente nel terminale
        user_message = input("\n[TU]: ").strip()
        
        # COMANDO DI USCITA: Permette di chiudere il programma in modo pulito
        if user_message.lower() in ["exit", "quit"]: 
            print("[BOT]: Spegnimento moduli... A presto!")
            break
            
        # VALIDAZIONE: Evitiamo di sprecare calcoli per input insignificanti
        if len(user_message) < 3: 
            print("[BOT]: Scrivi qualcosa di più lungo, per favore.")
            continue

        # --- FASE 1: GESTIONE DELLA MEMORIA ---
        # Aggiungiamo il nuovo messaggio alla storia con l'etichetta 'User:'
        history_buffer.append(f"User: {user_message}")
        
        # SE IL BUFFER È PIENO: Rimuoviamo il ricordo più vecchio (indice 0).
        # Questo mantiene il prompt entro i limiti di 'Context Window' del modello.
        if len(history_buffer) > max_history: 
            history_buffer.pop(0)

        # --- FASE 2: COSTRUZIONE DEL PROMPT ---
        # Uniamo i messaggi della storia in un'unica stringa separata da invii
        # Aggiungiamo 'Assistant:' alla fine per 'invitare' il bot a rispondere.
        prompt = "\n".join(history_buffer) + "\nAssistant:"
        
        # --- FASE 3: TOKENIZZAZIONE ---
        # Trasformiamo la stringa di testo in 'Tensor' (vettori matematici) per PyTorch.
        # return_tensors="pt" indica proprio il formato PyTorch.
        inputs = tokenizer(prompt, return_tensors="pt")
        
        # --- FASE 4: GENERAZIONE (INFERENZA) ---
        # 'torch.no_grad()' disabilita i calcoli dei gradienti, rendendo tutto più veloce e leggero.
        with torch.no_grad():
            output_tokens = model.generate(
                **inputs, 
                max_new_tokens=50,       # Quante nuove parole vogliamo generare al massimo
                pad_token_id=tokenizer.eos_token_id,
                no_repeat_ngram_size=2   # Blocca la ripetizione fastidiosa di coppie di parole
            )
        
        # --- FASE 5: DECODIFICA E PULIZIA ---
        # Trasformiamo i numeri (ID) generati di nuovo in testo leggibile.
        full_text = tokenizer.decode(output_tokens[0], skip_special_tokens=True)
        
        # Estraggiamo solo l'ultima parte della stringa (quella dopo l'ultima etichetta Assistant:)
        # per evitare di mostrare all'utente tutta la storia precedente o codici tecnici.
        response = full_text.split("Assistant:")[-1].strip().split("\n")[0]

        # OUTPUT E AGGIORNAMENTO
        print(f"[BOT]: {response}")
        
        # Aggiungiamo la risposta del bot alla storia per il prossimo turno
        history_buffer.append(f"Assistant: {response}")

# ESECUZIONE
if __name__ == "__main__":
    chat_loop()

# ==============================================================================
# SPIEGAZIONE  DI QUELLO CHE ACCADE SOTTO TRACCIA
# 1. Il Tokenizer spezza la frase: "Ciao" -> [15496]. Questo numero punta a un 
#    vettore di 768 dimensioni nel modello.
# 2. Il Modello (Transformer) analizza i rapporti tra questi numeri (Attention).
# 3. La tecnica 'Causal LM' (Language Modeling) cerca il numero (parola) che ha la 
#    probabilità statistica più alta di apparire dopo la parola 'Assistant:'.
# 4. Lo Sliding Window impedisce errori di "Out of Memory" (OOM) assicurando 
#    che il prompt non diventi infinitamente lungo.
# ==============================================================================


[STEP 1]: Risveglio dei neuroni (Caricamento GPT-2 via Hugging Face)...


OSError: gpt3 is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`